# Corrected analysis: Adult income

## Question and estimand

Describe associations between recorded characteristics and income above $50,000 in this Adult data extract. The estimand is descriptive for the unweighted rows in this extract. It is not a design-weighted estimate for the U.S. population and does not estimate causal effects.

## Corrected reporting choices

Report effect measures by type rather than treating Cramer's V and absolute rank-biserial effects as a common scale. Include all 14 tested features and show both adjusted p-value decisions, including the non-significant `fnlwgt` result. Display returned-zero p-values as underflow, not as exact probability zero. For missingness, compare high-income proportions and report Newcombe intervals for missing minus observed records.

For prediction, retain the stated stratified random 75/25 split and report the AUC with a test-set bootstrap interval. Do not describe this internal holdout as external validation. `fnlwgt` is a survey-related final weight, but applying it descriptively does not reconstruct a full complex survey design.

## Findings

The extract contains 48,842 records with 23.93% above $50,000; the `fnlwgt`-weighted prevalence is 23.78%, a descriptive sensitivity check only. Thirteen of 14 feature tests remain significant after both BH and Bonferroni adjustment; `fnlwgt` is not significant. The independently reproduced holdout AUC is approximately 0.899.

Missing workclass and occupation records have substantially lower high-income rates. Native-country missingness differs slightly in the opposite direction, with an interval including zero. These group differences are descriptive and may reflect collection or response processes; they do not establish that missingness caused income differences.

## Corrected estimates

The code cell prints aggregate subgroup proportions with Wilson intervals, all feature effects and adjusted decisions, missingness intervals, prevalence, and the internal holdout AUC. No row-level records are displayed.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'audit_tools.py').is_file())
sys.path.insert(0, str(project_root / 'src'))
from adult_audit import audit_adult_income

audit = audit_adult_income()
results_path = project_root / 'cases' / '05_multiple_comparisons' / 'results.json'
reference = json.loads(results_path.read_text(encoding='utf-8'))['reference_results']
subgroup_rows = []
for group_name, rows in reference['group_rates'].items():
    for row in rows:
        subgroup_rows.append({
            'group': group_name,
            'category': row['group'],
            'n': row['n'],
            'income > $50K': row['rate'],
            'Wilson 95% CI': (row['ci_95_lower'], row['ci_95_upper']),
        })
display(pd.DataFrame(subgroup_rows).round({'income > $50K': 4}))

feature_rows = audit['feature_associations']['results_sorted_by_reported_effect']
display(pd.DataFrame([
    {
        'feature': row['feature'],
        'type': row['type'],
        'effect metric': row['effect_name'],
        'effect': row['effect'],
        'BH significant': row['p_bh'] < 0.05,
        'Bonferroni significant': row['p_bonferroni'] < 0.05,
        'raw p-value': '0.0 (underflow)' if row['p_value'] == 0 else f"{row['p_value']:.3g}",
    }
    for row in feature_rows
]).round({'effect': 3}))

display(pd.DataFrame([
    {
        'feature': feature,
        'missing rate': summary['missing']['rate'],
        'observed rate': summary['observed']['rate'],
        'difference (pp)': 100 * summary['missing_minus_observed_income_rate'],
        'Newcombe 95% CI (pp)': tuple(round(100 * x, 2) for x in summary['difference_ci_95']),
    }
    for feature, summary in audit['missingness'].items()
]).round({'missing rate': 4, 'observed rate': 4, 'difference (pp)': 2}))

print({
    'n': audit['n_total'],
    'high_income_prevalence': audit['positive_prevalence'],
    'fnlwgt_weighted_prevalence_descriptive_only': audit['fnlwgt_weighted_prevalence'],
    'AUC': audit['adjusted_model']['test_auc'],
    'AUC_bootstrap_95_CI': audit['adjusted_model']['test_auc_ci_95'],
})

,group,category,n,income > $50K,Wilson 95% CI
0,age_band,25-34,12577,0.1682,"(0.1617291484744368, 0.17480171654590398)"
1,age_band,35-44,12193,0.3329,"(0.324585131689032, 0.34131197669282853)"
2,age_band,45-54,8771,0.3939,"(0.383734680814254, 0.4041817153128493)"
3,age_band,55+,6869,0.2861,"(0.27550252329992114, 0.29687230629431466)"
4,age_band,under-25,8432,0.0110,"(0.009012280007647979, 0.013491872037811803)"
5,education,10th,1389,0.0626,"(0.05105912967422313, 0.07662335543163437)"
6,education,11th,1812,0.0508,"(0.041580965975667274, 0.061864991769282875)"
7,education,12th,657,0.0731,"(0.05554555356353257, 0.09553676351576568)"
8,education,1st-4th,247,0.0324,"(0.016501455322103526, 0.06259814373929865)"
9,education,5th-6th,509,0.0530,"(0.036708890267067876, 0.07607734771319134)"


,feature,type,effect metric,effect,BH significant,Bonferroni significant,raw p-value
0,relationship,categorical,Cramer's V,0.454,True,True,0.0 (underflow)
1,marital-status,categorical,Cramer's V,0.448,True,True,0.0 (underflow)
2,education-num,numeric,absolute rank-biserial,0.432,True,True,0.0 (underflow)
3,education,categorical,Cramer's V,0.366,True,True,0.0 (underflow)
4,age,numeric,absolute rank-biserial,0.365,True,True,0.0 (underflow)
5,occupation,categorical,Cramer's V,0.350,True,True,0.0 (underflow)
6,hours-per-week,numeric,absolute rank-biserial,0.344,True,True,0.0 (underflow)
7,sex,categorical,Cramer's V,0.215,True,True,0.0 (underflow)
8,workclass,categorical,Cramer's V,0.182,True,True,0.0 (underflow)
9,capital-gain,numeric,absolute rank-biserial,0.179,True,True,0.0 (underflow)


,feature,missing rate,observed rate,difference (pp),Newcombe 95% CI (pp)
0,workclass,0.0947,0.2481,-15.34,"(-16.44, -14.13)"
1,occupation,0.0943,0.2481,-15.38,"(-16.48, -14.18)"
2,native-country,0.2567,0.2390,1.77,"(-1.06, 4.83)"


{'n': 48842, 'high_income_prevalence': 0.23928176569346055, 'fnlwgt_weighted_prevalence_descriptive_only': 0.2377759609645643, 'AUC': 0.8989270610642558, 'AUC_bootstrap_95_CI': [0.8931921538635177, 0.9050196293956572]}
